# InfraScan System Architecture

This notebook describes a practical, beginner-friendly technical architecture for **InfraScan - AI-Powered Crack & Damage Assessment for Civil Infrastructure**.

The architecture below is a design proposal. The existing project includes dataset-analysis and planning documentation, but the application components described here are planned unless separately implemented and verified.

## 1. System architecture overview

InfraScan is planned as a web application with a browser-based frontend, a backend API, object storage for original and processed images, a relational database for users and analysis records, and a model-serving layer for AI predictions.

The first implementation should use a modular monolith: one backend application with clear modules is easier to understand and deploy than many independent microservices. Model inference can initially run in the backend process or a separate worker, then be separated when usage or hardware requirements justify it.

### Planned architecture diagram

```mermaid
flowchart LR
    User[Inspector / Engineer / Maintenance Team]
    Browser[Web Frontend]
    API[Backend API]
    Auth[Authentication and RBAC]
    Queue[Job Queue - optional for long inference]
    Preprocess[Image Validation and Preprocessing]
    Models[AI Model Service]
    Store[(Object Storage)]
    DB[(Relational Database)]
    Report[Results and Report Service]

    User --> Browser
    Browser --> API
    API --> Auth
    API --> Store
    API --> DB
    API --> Queue
    Queue --> Preprocess
    Preprocess --> Models
    Models --> Store
    Models --> DB
    API --> Report
    Report --> Store
    Report --> Browser
```

The arrows represent planned responsibilities and data movement, not a claim that these services are currently running.

## 2. Frontend architecture and responsibilities

### Recommended approach

Use a React and TypeScript frontend, preferably with a simple Vite setup for the first version. This provides component reuse, type checking, and a large ecosystem without requiring a complicated frontend platform.

### Planned frontend responsibilities

- Provide login and logout screens.
- Allow an authorized user to select an image and submit it for analysis.
- Show upload progress, validation messages, and processing status.
- Display the original image beside classification, detection, or segmentation results.
- Show confidence values and warnings in plain language.
- Present analysis history according to the user's role and permissions.
- Allow a report or result summary to be downloaded when that feature is implemented.

The frontend should not contain secret credentials or make model decisions itself. It should call documented backend endpoints and treat all results as untrusted server data that must be rendered safely.

## 3. Backend architecture and responsibilities

Use a Python backend API, such as FastAPI, because it provides clear request validation, automatic API documentation, and good support for Python-based image and machine-learning libraries.

### Planned backend modules

- **Authentication module:** Verifies identity and creates or validates sessions or tokens.
- **User and role module:** Applies permissions for inspectors, engineers, maintenance users, and administrators.
- **Upload module:** Validates metadata, receives files or signed upload references, and records an upload.
- **Analysis module:** Creates an analysis job and tracks its status from queued to completed or failed.
- **Preprocessing module:** Performs the documented image preparation steps.
- **Inference module:** Selects a versioned model and requests classification, detection, or segmentation predictions.
- **Results module:** Stores prediction metadata, confidence values, overlays, and review status.
- **Reporting module:** Builds a human-readable result summary without hiding uncertainty.

The backend should enforce validation and authorization even when the frontend has already performed similar checks. Frontend checks improve usability; backend checks protect the system.

## 4. Image upload and storage architecture

Store image bytes in object storage rather than in database rows. A local filesystem can be used during early development, while an S3-compatible service such as Amazon S3, MinIO, or another approved object store is more suitable for a deployed system.

### Planned upload sequence

1. The frontend asks the backend to start an upload.
2. The backend authenticates the user and creates an upload record with a generated identifier.
3. The file is uploaded through a controlled endpoint or a short-lived signed URL.
4. The backend checks the declared and detected file type, size, dimensions, and readability.
5. The original file is stored as read-only source evidence. Processed images and visual overlays use separate object keys.
6. A database record links the storage keys to the user, project, inspection, and analysis job.

Object names should use generated identifiers rather than user-supplied filenames. Upload limits, content checks, malware scanning where required, and lifecycle policies should be defined before production deployment.

## 5. Image preprocessing pipeline

Preprocessing must be deterministic and consistent between training and inference. A planned pipeline is:

1. Confirm that the input is an allowed image format and can be decoded.
2. Check maximum file size, pixel dimensions, and image mode.
3. Preserve the original image and record its metadata.
4. Convert to the color representation expected by the selected model, such as RGB.
5. Resize or pad to the model's required input dimensions while recording the transformation.
6. Normalize pixel values using the same parameters used during training.
7. Optionally apply carefully controlled quality checks for blur, extreme darkness, or insufficient resolution.
8. Pass the prepared tensor to inference and retain the transformation metadata for mapping results back to the original image.

Training augmentation, such as rotation or brightness changes, should be applied only as part of the training pipeline and not automatically treated as an inspection result. The original image must remain available for human review.

## 6. AI/ML model integration

The AI layer should expose a stable inference interface so that models can be replaced without redesigning the rest of the application. Each prediction should record the model name, model version, preprocessing version, timestamp, and confidence or uncertainty information.

### Classification

A binary classifier can first estimate whether an image or image region contains a visible crack. The existing concrete-crack image dataset is suitable for investigating a classification baseline, but performance must be evaluated on held-out data and representative real inspection images.

### Object detection

Detection models can identify likely crack locations with bounding boxes or related regions. This requires appropriately labeled location data. A classification-only dataset does not automatically provide the annotations needed to train and validate detection accurately.

### Segmentation

Segmentation models can estimate crack pixels or boundaries. This requires reliable pixel-level masks and separate evaluation. Segmentation is therefore a planned later stage, not an automatic result of adding a classifier.

For an initial implementation, model inference can be a backend module with a versioned model file. Long-running inference should later move to a worker queue so that the API remains responsive. No classification, detection, or segmentation service is claimed to be implemented by this architecture document.

## 7. Database architecture

Use a relational database such as PostgreSQL for structured records and relationships. It is a practical choice for users, roles, uploads, inspections, jobs, predictions, and audit events.

### Suggested logical entities

- `users`: identity and account status.
- `roles` and `user_roles`: role assignments and permissions.
- `projects` or `assets`: the infrastructure context for an inspection.
- `inspections`: inspection date, location, notes, and responsible user.
- `uploads`: original filename metadata, storage key, checksum, dimensions, and validation status.
- `analysis_jobs`: processing status, requested model task, timestamps, and failure information.
- `model_versions`: model identifiers, versions, and evaluation references.
- `predictions`: classification values, detections, segmentation references, confidence, and review status.
- `reports`: generated report metadata and storage key.
- `audit_events`: security-sensitive actions and state changes.

Large images and overlays should remain in object storage; the database should store references and searchable metadata. Database migrations and backups should be part of deployment practice.

## 8. Authentication and role-based access

Authentication confirms who a user is; authorization determines what that user may do. The backend must enforce both.

### Suggested roles

- **Inspector:** Upload images, start analyses, and view assigned inspection results.
- **Civil engineer:** Review results, add technical notes, and access permitted inspection history.
- **Maintenance team member:** View assigned results and update operational follow-up status.
- **Administrator:** Manage users, roles, system configuration, and audit access.

Use a standards-based identity provider or a well-maintained authentication library rather than implementing password security from scratch. Use secure, expiring sessions or tokens, password hashing where applicable, multi-factor authentication when appropriate, and server-side ownership checks for every protected resource. The exact provider and token strategy remain implementation decisions.

## 9. Complete end-to-end data flow

1. A user signs in through the frontend.
2. The backend authenticates the user and returns an authenticated session.
3. The user selects an image and optionally associates it with an asset or inspection.
4. The backend validates authorization and creates an upload record.
5. The image is transferred to object storage and checked for type, size, dimensions, and readability.
6. The backend creates an analysis job and records the selected task and model version.
7. A worker or backend process loads the source image, applies the versioned preprocessing pipeline, and runs the requested model.
8. Predictions, confidence information, transformation metadata, and generated overlays are stored in the appropriate locations.
9. The frontend polls or subscribes to the job status and requests the authorized results.
10. The user reviews the output, warnings, and original image; a qualified professional remains responsible for engineering interpretation.
11. The system can generate a report and record relevant review or follow-up actions.

Failures should update the job with a clear error state and should not produce a success-shaped result. Audit events should capture important access and state transitions without storing unnecessary sensitive data.

## 10. Technology stack and reasons for each selection

| Layer | Practical recommendation | Reason |
| --- | --- | --- |
| Frontend | React + TypeScript + Vite | Component-based UI, type safety, fast local development, and a large ecosystem. |
| Styling | A small accessible CSS system or a utility framework | Consistent layouts and responsive screens without building every style from scratch. |
| Backend API | Python + FastAPI | Clear request validation, automatic API documentation, and direct access to Python image/ML libraries. |
| Image processing | Pillow initially; OpenCV when computer-vision operations require it | Pillow is simple for file checks and metadata; OpenCV adds mature image operations when needed. |
| ML framework | PyTorch | Flexible model experimentation and broad support for vision models. |
| Database | PostgreSQL | Reliable relational data, transactions, indexes, and clear relationships. |
| Object storage | S3-compatible storage | Designed for large files, access policies, lifecycle rules, and scalable storage. |
| Background jobs | Redis plus a Python task worker, or a managed queue | Keeps longer inference work away from synchronous API requests. This can be deferred for a small prototype. |
| Deployment | Docker for repeatable environments | Makes local, test, and deployment environments more consistent. |
| Testing | Pytest for backend and model utilities; frontend component tests | Supports focused automated checks around validation, permissions, and UI behavior. |

These are recommendations for implementation, not a record of technologies already configured in InfraScan. The simplest first version can omit the queue and use local storage during development, provided that these substitutions are documented.

## 11. Security, scalability, and reliability considerations

### Security

- Enforce authentication and object-level authorization on every protected API request.
- Validate file type by inspecting content as well as trusting extensions; limit file size and image dimensions.
- Use generated storage keys, private buckets, short-lived signed URLs, and encryption in transit and at rest.
- Keep secrets in environment configuration or a secret manager, never in notebooks or source control.
- Protect against common web risks such as injection, cross-site request forgery where relevant, unsafe file rendering, and excessive request rates.
- Log security-relevant events without logging raw credentials or unnecessary personal information.

### Scalability

- Keep image bytes out of the relational database.
- Use asynchronous jobs for expensive model inference and allow multiple workers when demand grows.
- Version models and preprocessing so new versions can be evaluated without silently changing old results.
- Add indexes for common queries such as user, project, inspection, and job status.
- Add caching only after measuring a real bottleneck.

### Reliability

- Use explicit job states such as queued, processing, completed, failed, and cancelled.
- Make retries safe by avoiding duplicate records or duplicate report generation.
- Back up the database and define object-storage retention and recovery procedures.
- Monitor API errors, queue delays, storage failures, and model inference failures.
- Test invalid images, interrupted uploads, permission failures, model errors, and partial service outages.
- Display uncertainty and limitations so that users do not mistake an automated prediction for a guaranteed engineering conclusion.

The architecture is intended to make these concerns visible early. Detailed threat modeling, capacity targets, deployment configuration, and operational procedures are future implementation work.

## Implementation status boundary

This notebook is an architecture design document. It does not claim that the frontend, backend API, storage service, database, authentication system, job queue, model-serving layer, or reporting service is currently implemented. Existing dataset exploration can inform the preprocessing and classification work, while the other components require future development, testing, and validation before they can be described as working.